# Alt i et

In [3]:
import numpy as np
import nibabel as nib
from skimage.measure import marching_cubes
from scipy.ndimage import binary_closing, binary_fill_holes
import trimesh


# ============================================================
# SETTINGS 
# ============================================================

# 0 = no closing
# Try 1 later if desired
CLOSING_RADIUS = 2

LABEL_FILE = "Brain_label.nii"

OUTPUT_OBJ = f"brain_segmented_{CLOSING_RADIUS:02d}.obj"



# Only these are real anatomical labels
MIN_LABEL = 1
MAX_LABEL = 86


# ============================================================
# LOAD NIFTI
# ============================================================

nii = nib.load(LABEL_FILE)
labels = nii.get_fdata()

print("Volume shape:", labels.shape)


# ============================================================
# CREATE BINARY BRAIN MASK
# ============================================================

# Labels 1–86 = brain
# Label 0 = air
# Labels 87–90 = stray voxels and excluded

binary_mask = (
    (labels >= MIN_LABEL)
    & (labels <= MAX_LABEL)
)

print("Brain voxels:", binary_mask.sum())


# ============================================================
# OPTIONAL MORPHOLOGICAL CLOSING
# ============================================================

if CLOSING_RADIUS > 0:

    structure_size = 2 * CLOSING_RADIUS + 1

    structure = np.ones(
        (
            structure_size,
            structure_size,
            structure_size
        ),
        dtype=bool
    )

    binary_mask = binary_closing(
        binary_mask,
        structure=structure
    )

    print(
        "Brain voxels after closing:",
        binary_mask.sum()
    )

# ============================================================
# FILL HOLES
# ============================================================

binary_mask = binary_fill_holes(
    binary_mask,
    structure=np.ones((3, 3, 3), dtype=bool),
)

print("Brain voxels after filling enclosed holes:", binary_mask.sum())

# ============================================================
# MARCHING CUBES
# ============================================================

print("Running marching cubes...")

verts_voxel, faces, normals_voxel, values = marching_cubes(
    binary_mask.astype(np.uint8),
    level=0.5
)

print("Vertices:", len(verts_voxel))
print("Faces:", len(faces))


# ============================================================
# CONVERT VERTICES TO WORLD COORDINATES
# ============================================================

verts_world = nib.affines.apply_affine(
    nii.affine,
    verts_voxel
)

mesh = trimesh.Trimesh(
    vertices=verts_world,
    faces=faces,
    process=False
)

mesh.export(
    OUTPUT_OBJ,
)

print()
print("Saved:")
print(" ", OUTPUT_OBJ)
print(f"faces: {mesh.faces.shape}")
print(f"vertices: {mesh.vertices.shape}")

Volume shape: (149, 208, 140)
Brain voxels: 814476
Brain voxels after closing: 934914
Brain voxels after filling enclosed holes: 944982
Running marching cubes...
Vertices: 102728
Faces: 205472

Saved:
  brain_segmented_02.obj
faces: (205472, 3)
vertices: (102728, 3)


# Hele processen

In [ ]:
import numpy as np
import nibabel as nib
from skimage.measure import marching_cubes
from scipy.ndimage import binary_closing
import trimesh


# ============================================================
# SETTINGS 
# ============================================================

# 0 = no closing
# Try 1 later if desired
CLOSING_RADIUS = 2

LABEL_FILE = "Brain_label.nii"


# Only these are real anatomical labels
MIN_LABEL = 1
MAX_LABEL = 86


# ============================================================
# LOAD NIFTI
# ============================================================

nii = nib.load(LABEL_FILE)
labels = nii.get_fdata()

print("Volume shape:", labels.shape)


# ============================================================
# CREATE BINARY BRAIN MASK
# ============================================================

# Labels 1–86 = brain
# Label 0 = air
# Labels 87–90 = stray voxels and excluded

binary_mask = (
    (labels >= MIN_LABEL)
    & (labels <= MAX_LABEL)
)

print("Brain voxels:", binary_mask.sum())


# ============================================================
# OPTIONAL MORPHOLOGICAL CLOSING
# ============================================================

if CLOSING_RADIUS > 0:

    structure_size = 2 * CLOSING_RADIUS + 1

    structure = np.ones(
        (
            structure_size,
            structure_size,
            structure_size
        ),
        dtype=bool
    )

    binary_mask = binary_closing(
        binary_mask,
        structure=structure
    )

    print(
        "Brain voxels after closing:",
        binary_mask.sum()
    )

OUTPUT_NIFTI = f"/Users/jacob/Desktop/Uni/7_semester/Bachelor/Master_mesh/brain_mask_{CLOSING_RADIUS:02d}.nii"

mask_nii = nib.Nifti1Image(
    binary_mask.astype(np.uint8),
    nii.affine,
    nii.header
)

mask_nii.set_data_dtype(np.uint8)
nib.save(mask_nii, OUTPUT_NIFTI)

print("Saved NIfTI mask:", OUTPUT_NIFTI)

Volume shape: (149, 208, 140)
Brain voxels: 814476
Brain voxels after closing: 934914
Saved NIfTI mask: /Users/jacob/Desktop/Uni/7_semester/Bachelor/Master_mesh/brain_mask_02.nii


# lav paintbrush segmentering

In [ ]:
ORIGINAL_MASK_FILE = "brain_mask_02.nii"
EDITED_SEGMENTATION_FILE = "brain_mask_02_segmentation.nii.gz"
OUTPUT_NIFTI = "brain_mask_02_with_edits_closed_r02.nii"

CLOSING_RADIUS = 2

# Load the original binary mask and your ITK-SNAP-edited segmentation
original_nii = nib.load(ORIGINAL_MASK_FILE)
edited_nii = nib.load(EDITED_SEGMENTATION_FILE)

original_mask = original_nii.get_fdata() > 0

# Assumes the areas you painted in ITK-SNAP have label value 1.
painted_mask = edited_nii.get_fdata() == 1

# Include both the old mask and your newly painted voxels
combined_mask = original_mask | painted_mask

# Morphological closing with radius 2
structure_size = 2 * CLOSING_RADIUS + 1
structure = np.ones(
    (structure_size, structure_size, structure_size),
    dtype=bool,
)

closed_mask = binary_closing(
    combined_mask,
    structure=structure,
)

# Save as 0/1 uint8 NIfTI, using the original mask's spatial metadata
output_nii = nib.Nifti1Image(
    closed_mask.astype(np.uint8),
    original_nii.affine,
    original_nii.header,
)
output_nii.set_data_dtype(np.uint8)

nib.save(output_nii, OUTPUT_NIFTI)
print("Saved:", OUTPUT_NIFTI)
print("Voxels before closing:", combined_mask.sum())
print("Voxels after closing:", closed_mask.sum())


Saved: brain_mask_02_with_edits_closed_r02.nii
Voxels before closing: 942933
Voxels after closing: 943825


# Marching cubes and the rest

In [ ]:
EDITED_NIFTI = "brain_mask_02_with_edits_closed_r02.nii"
OUTPUT_OBJ = "Brain_segmented_02_with_edits_final.obj"

mask = nib.load(EDITED_NIFTI)

original_mask = original_nii.get_fdata()

# ============================================================
# MARCHING CUBES
# ============================================================

print("Running marching cubes...")

verts_voxel, faces, normals_voxel, values = marching_cubes(
    binary_mask.astype(np.uint8),
    level=0.5
)

print("Vertices:", len(verts_voxel))
print("Faces:", len(faces))


# ============================================================
# CONVERT VERTICES TO WORLD COORDINATES
# ============================================================

verts_world = nib.affines.apply_affine(
    nii.affine,
    verts_voxel
)

mesh = trimesh.Trimesh(
    vertices=verts_world,
    faces=faces,
    process=False
)

mesh.export(
    OUTPUT_OBJ,
)

print()
print("Saved:")
print(" ", OUTPUT_OBJ)
print(f"faces: {mesh.faces.shape}")
print(f"vertices: {mesh.vertices.shape}")